In [ ]:
# Optional install; works from root or provider directory.
from pathlib import Path

_install_root = next(
    p
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "notebooks" / "qwen" / "requirements.txt").is_file()
)
_requirements = str(_install_root / "notebooks" / "qwen" / "requirements.txt")
%pip install -r "$_requirements" -q

# Text-to-SQL

# Dynamic Text-to-SQL — Hugging Face only

The reusable `notebook_analytics.py` module discovers the actual snapshot schema,
validates generated SQL, executes it read-only, and handles arbitrary result columns
and multiple rows. There are no question-specific SQL templates or fixed response fields.
The same HF model generates SQL and separately writes Vietnamese answers.

Run from this repository directory. Configure an untracked `.env` or environment:

```text
HF_TOKEN=<your token>
HF_MODEL_ID=Qwen/Qwen3-4B-Instruct-2507
HF_PROVIDER=auto
HF_AGENT_MAX_STEPS=3
HF_MAX_TOKENS=1024
HF_RESPONSE_MAX_TOKENS=1500
HF_FEW_SHOT=true
HF_REQUEST_TIMEOUT_SECONDS=30
HF_SQL_RESULT_ROW_LIMIT=100
HF_SQL_RESULT_BYTE_LIMIT=64000
HF_SQL_TIMEOUT_SECONDS=3
SHOW_SQL_EVIDENCE=false
```

Optional `HF_SQL_ALLOWED_TABLES` is a comma-separated subset of actual snapshot tables.
By default all non-system tables in this disposable snapshot are allowed. It is not
a connection to arbitrary server tables. Existing BENCHMARK settings are unchanged.

Run loading cells only when you want a database refresh. They preserve the existing
MSSQL-to-SQLite loading behavior. For inference-only testing on an existing snapshot,
run setup/configuration, schema initialization, Cell A, and Cell B; skip load cells.
Schema initialization can locate BENCHMARK_SQLITE_PATH without the loader.

Result checks enforce references and literal values but do not prove business-semantic
correctness. Empty/truncated results or rejected/failed model output use a labelled table
fallback. Unknown freshness is reported as unknown, never inferred from file mtime.
See NOTEBOOK_ANALYTICS.md for setup, semantics, tests, limitations and rerun instructions.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "notebooks" / "shared" / "analytics.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the self-healthy-kafka repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

Load the latest local `poc-mssql` snapshot into SQLite:

In [ ]:
from datetime import date, datetime, timezone
from decimal import Decimal
from uuid import UUID

import pyodbc
from sqlalchemy import Column, Integer, MetaData, String, Table, Text, create_engine, insert, text


def local_poc_connection_string():
    """Build a local-only connection string without persisting the SA password."""
    import subprocess

    completed = subprocess.run(
        ["docker", "inspect", "poc-mssql", "--format", "{{range .Config.Env}}{{println .}}{{end}}"],
        check=True,
        capture_output=True,
        text=True,
    )
    password_line = next(
        (
            line
            for line in completed.stdout.splitlines()
            if line.startswith(("MSSQL_SA_PASSWORD=", "SA_PASSWORD="))
        ),
        None,
    )
    if not password_line:
        raise RuntimeError("Local poc-mssql password environment variable was not found.")
    password = password_line.split("=", 1)[1]
    return (
        "DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost,14330;"
        "DATABASE=ingest_reference;UID=sa;PWD=" + password + ";TrustServerCertificate=yes"
    )


snapshot_refresh_metadata = {
    "started_at": datetime.now(timezone.utc).isoformat(),
    "source": "configured MSSQL source",
    "tables": {},
}

source_connection_string = os.getenv("BENCHMARK_MSSQL_CONNECTION_STRING")
if not source_connection_string:
    source_connection_string = local_poc_connection_string()
    print("Using the local poc-mssql snapshot source.")

available_drivers = pyodbc.drivers()
if "ODBC Driver 17 for SQL Server" not in available_drivers:
    raise RuntimeError(
        "ODBC Driver 17 for SQL Server is required for local poc-mssql. "
        f"Installed drivers: {available_drivers}"
    )

snapshot_path = Path(os.getenv("BENCHMARK_SQLITE_PATH", "self_healthy_kafka_snapshot.db"))
if not snapshot_path.is_absolute():
    snapshot_path = REPO_ROOT / snapshot_path
engine = create_engine(f"sqlite:///{snapshot_path}")
metadata_obj = MetaData()

connector_healing_queue = Table(
    "ConnectorHealingQueue",
    metadata_obj,
    Column("QueueId", String(36), primary_key=True),
    Column("RootConnectorName", String(255), nullable=False),
    Column("CurrentConnectorName", String(255), nullable=False),
    Column("ConnectorClass", String(500)),
    Column("HealingMode", String(50), nullable=False),
    Column("QueueStatus", String(50), nullable=False),
    Column("FinalOutcome", String(50)),
    Column("ReceivedAt", String(40), nullable=False),
    Column("StartedAt", String(40)),
    Column("CompletedAt", String(40)),
    Column("NextAttemptAt", String(40)),
    extend_existing=True,
)


def normalize(value):
    if isinstance(value, (UUID, datetime, date, Decimal)):
        return str(value)
    return value


def fetch_rows(cursor, query, columns):
    cursor.execute(query)
    return [
        dict(zip(columns, (normalize(value) for value in row), strict=True))
        for row in cursor.fetchall()
    ]


redact_raw_log_text = os.getenv("BENCHMARK_REDACT_RAW_LOG_TEXT", "true").lower() != "false"
try:
    source_connection = pyodbc.connect(source_connection_string)
except pyodbc.InterfaceError as exc:
    raise RuntimeError(
        "Cannot resolve the ODBC driver in BENCHMARK_MSSQL_CONNECTION_STRING. "
        "Use DRIVER={ODBC Driver 17 for SQL Server}; not Driver 18 or a DSN name. "
        f"Installed drivers: {available_drivers}"
    ) from exc

with source_connection:
    cursor = source_connection.cursor()
    queue_columns = [
        "QueueId",
        "RootConnectorName",
        "CurrentConnectorName",
        "ConnectorClass",
        "HealingMode",
        "QueueStatus",
        "FinalOutcome",
        "ReceivedAt",
        "StartedAt",
        "CompletedAt",
        "NextAttemptAt",
    ]
    snapshot_rows = {
        "ConnectorHealingQueue": fetch_rows(
            cursor,
            """
            SELECT CAST([QueueId] AS varchar(36)), [RootConnectorName], [CurrentConnectorName],
                   [ConnectorClass], [HealingMode], [QueueStatus], [FinalOutcome],
                   CONVERT(varchar(40), [ReceivedAt], 127), CONVERT(varchar(40), [StartedAt], 127),
                   CONVERT(varchar(40), [CompletedAt], 127), CONVERT(varchar(40), [NextAttemptAt], 127)
            FROM [dbo].[ConnectorHealingQueue]
            ORDER BY [ReceivedAt] DESC
        """,
            queue_columns,
        ),
    }

metadata_obj.drop_all(engine, checkfirst=True)
metadata_obj.create_all(engine)
with engine.begin() as connection:
    if snapshot_rows["ConnectorHealingQueue"]:
        connection.execute(insert(connector_healing_queue), snapshot_rows["ConnectorHealingQueue"])

print(
    f"Loaded {len(snapshot_rows['ConnectorHealingQueue'])} latest queue rows into {snapshot_path}."
)
snapshot_refresh_metadata["tables"]["ConnectorHealingQueue"] = {
    "loaded_at": datetime.now(timezone.utc).isoformat(),
    "rows": len(snapshot_rows["ConnectorHealingQueue"]),
}

### Build our agent

The loader retains its existing physical queue schema. The model context is discovered from the actual snapshot after both tables are available.


In [ ]:
# Schema inspection is centralized in the schema-initialization cell below.


Import the reusable parser, read-only executor and generic result/response workflow.


In [ ]:
import importlib

import notebooks.shared.analytics as notebook_analytics

importlib.reload(notebook_analytics)
from notebooks.shared.analytics import Snapshot, Workflow  # noqa: E402 - reload first

Configure one Hugging Face model for both stages. Credentials are read only from the environment; no API call happens in this cell.


In [ ]:
from huggingface_hub import InferenceClient

hf_token = os.getenv("HF_TOKEN", "").strip()
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in an untracked .env file or environment, then rerun setup.")
hf_model_id = os.getenv("HF_MODEL_ID", "Qwen/Qwen3-4B-Instruct-2507").strip()
hf_provider = os.getenv("HF_PROVIDER", "auto").strip().lower() or "auto"
hf_client = InferenceClient(
    model=hf_model_id,
    provider=hf_provider,
    api_key=hf_token,
    timeout=int(os.getenv("HF_REQUEST_TIMEOUT_SECONDS", "30")),
)

### Level 2: Table joins

Load the second local snapshot table, `ConnectorHealingLogs`, so the original join scenario remains available.

In [ ]:
connector_healing_logs = Table(
    "ConnectorHealingLogs",
    metadata_obj,
    Column("Id", String(36), primary_key=True),
    Column("QueueId", String(36), nullable=False),
    Column("ConnectorName", String(255), nullable=False),
    Column("EventType", String(100), nullable=False),
    Column("AttemptNo", Integer),
    Column("HealingStep", Integer),
    Column("Severity", String(50), nullable=False),
    Column("Message", Text),
    Column("Details", Text),
    Column("CreatedAt", String(40), nullable=False),
    extend_existing=True,
)

with pyodbc.connect(source_connection_string) as source_connection:
    cursor = source_connection.cursor()
    log_columns = [
        "Id",
        "QueueId",
        "ConnectorName",
        "EventType",
        "AttemptNo",
        "HealingStep",
        "Severity",
        "Message",
        "Details",
        "CreatedAt",
    ]
    message_expression = "N'[REDACTED]'" if redact_raw_log_text else "[Message]"
    details_expression = "N'[REDACTED]'" if redact_raw_log_text else "[Details]"
    snapshot_rows["ConnectorHealingLogs"] = fetch_rows(
        cursor,
        f"""
        SELECT CAST([Id] AS varchar(36)), CAST([QueueId] AS varchar(36)), [ConnectorName],
               [EventType], [AttemptNo], [HealingStep], [Severity],
               {message_expression}, {details_expression}, CONVERT(varchar(40), [CreatedAt], 127)
        FROM [dbo].[ConnectorHealingLogs]
        ORDER BY [CreatedAt] DESC
    """,
        log_columns,
    )

# Idempotent refresh: clear the previous local snapshot before loading current rows.
metadata_obj.create_all(engine, checkfirst=True)
with engine.begin() as connection:
    connection.execute(text('DELETE FROM "ConnectorHealingLogs"'))
    if snapshot_rows["ConnectorHealingLogs"]:
        connection.execute(insert(connector_healing_logs), snapshot_rows["ConnectorHealingLogs"])

print(f"Loaded {len(snapshot_rows['ConnectorHealingLogs'])} latest log rows.")
snapshot_refresh_metadata["tables"]["ConnectorHealingLogs"] = {
    "loaded_at": datetime.now(timezone.utc).isoformat(),
    "rows": len(snapshot_rows["ConnectorHealingLogs"]),
}
snapshot_refresh_metadata["finished_at"] = datetime.now(timezone.utc).isoformat()

Discover schema and initialize a fresh workflow after loading, or against an existing snapshot. Rerun this cell after a refresh or schema/configuration change.


In [ ]:
import json

snapshot_path = Path(os.getenv("BENCHMARK_SQLITE_PATH", "self_healthy_kafka_snapshot.db"))
if not snapshot_path.is_absolute():
    snapshot_path = REPO_ROOT / snapshot_path
allowed = os.getenv("HF_SQL_ALLOWED_TABLES", "").strip()
snapshot = Snapshot(
    snapshot_path,
    allowed_tables=[t.strip() for t in allowed.split(",") if t.strip()] if allowed else None,
    row_limit=int(os.getenv("HF_SQL_RESULT_ROW_LIMIT", "100")),
    byte_limit=int(os.getenv("HF_SQL_RESULT_BYTE_LIMIT", "64000")),
    timeout_seconds=float(os.getenv("HF_SQL_TIMEOUT_SECONDS", "3")),
    refresh_metadata=globals().get("snapshot_refresh_metadata"),
)
workflow = Workflow(
    snapshot,
    hf_client,
    model_id=hf_model_id,
    provider=hf_provider,
    max_attempts=int(os.getenv("HF_AGENT_MAX_STEPS", "3")),
    sql_max_tokens=int(os.getenv("HF_MAX_TOKENS", "1024")),
    response_max_tokens=int(os.getenv("HF_RESPONSE_MAX_TOKENS", "1500")),
    few_shot=os.getenv("HF_FEW_SHOT", "true").strip().lower() in {"true", "1", "yes"},
)
print(json.dumps(snapshot.context(), ensure_ascii=False, indent=2))

## Two independently timed inference stages

**Cell A:** preserve/edit your question; ask HF for SQL or clarification, then execute
validated SQL. A bounded repair loop receives actual SQL errors, not fabricated facts.
**Cell B:** send the question, executed SQL, result rows and metric definitions to the
same HF model without tools. Failed checks return the verified table, not an invented answer.

For clarification, add your answer to the question in Cell A and rerun both stages.
For a new question, rerun Cell A before Cell B. State is cleared at the start of Cell A.


In [ ]:
import json

question = "Thời gian xử lý trung bình (tính bằng phút) từ lúc nhận (ReceivedAt) đến khi hoàn tất (CompletedAt) của các queue thành công là bao nhiêu?"

verified_result = None
final_answer = None
workflow.reset()
try:
    verified_result = workflow.query(question)
    if workflow.clarification:
        print("Clarification required:", workflow.clarification)
    else:
        print("Verified returned rows:", verified_result["returned_row_count"])
        print("Truncated:", verified_result["truncated"])
        print("Model interpretation (not independently verified):", workflow.interpretation)
        if os.getenv("SHOW_SQL_EVIDENCE", "false").lower() == "true":
            print(json.dumps(verified_result, ensure_ascii=False, indent=2))
finally:
    print("Step A — Hugging Face + SQLite:", json.dumps(workflow.metrics, ensure_ascii=False))
    if workflow.result is None and not workflow.clarification:
        print("No verified result. Diagnostics:", json.dumps(workflow.trace, ensure_ascii=False))

In [ ]:
import json

final_answer = None
if workflow.question != question:
    raise RuntimeError("Question changed; rerun Cell A before generating a response.")
final_answer = workflow.respond()
print("Response source:", final_answer["source"])
if final_answer.get("reason"):
    print("Fallback reason:", final_answer["reason"])
print(final_answer["text"])
if final_answer.get("scope"):
    print(final_answer["scope"])
print("Step B — Hugging Face response:", json.dumps(workflow.metrics, ensure_ascii=False))